In [24]:
from pathlib import Path
import pandas as pd


def parse_sbi_small_cap(filepath: str | Path) -> pd.DataFrame:
    """
    Parse SBI Small Cap Fund holdings from the SSCF worksheet.
    Returns a standardized DataFrame.
    """

    # -------------------------
    # Read Excel
    # -------------------------
    raw_df = pd.read_excel(
        filepath,
        sheet_name="SSCF",
        header=5
    )

    # -------------------------
    # Remove empty Excel columns
    # -------------------------
    raw_df = raw_df.loc[:, ~raw_df.columns.str.startswith("Unnamed")]

    # -------------------------
    # Keep only valid holdings
    # -------------------------
    raw_df = raw_df[
        raw_df["ISIN"]
        .astype(str)
        .str.match(r"^[A-Z]{2}[A-Z0-9]{10}$")
    ].copy()

    # -------------------------
    # Build standardized dataframe
    # -------------------------
    normalized_df = pd.DataFrame({
        "report_date": "2026-05-31",      # TODO: Extract automatically
        "fund_house": "SBI",
        "fund_name": "Small Cap",
        "company": raw_df["Name of the Instrument / Issuer"],
        "isin": raw_df["ISIN"],
        "industry": raw_df["Rating / Industry^"],
        "quantity": raw_df["Quantity"],
        "market_value_lakhs": raw_df["Market value\n(Rs. in Lakhs)"],
        "aum_percent": raw_df["% to AUM"],
    })

    return normalized_df


# -------------------------
# Test
# -------------------------
if __name__ == "__main__":

    filepath = Path("/Users/sridharshrawani/Projects/Sridhar/Equity-Research/backend/data/raw/May_2026/All-Schemes-Monthly-Portfolio---as-on-31st-May-2026.xlsx")

    df = parse_sbi_small_cap(filepath)

    print(df.head())
    print(df.info())

  report_date fund_house  fund_name  \
4  2026-05-31        SBI  Small Cap   
5  2026-05-31        SBI  Small Cap   
6  2026-05-31        SBI  Small Cap   
7  2026-05-31        SBI  Small Cap   
8  2026-05-31        SBI  Small Cap   

                                            company          isin  \
4                                 Ather Energy Ltd.  INE0LEZ01016   
5                 Navin Fluorine International Ltd.  INE048G01026   
6                   Honeywell Automation India Ltd.  INE671A01010   
7             Kalpataru Projects International Ltd.  INE220B01022   
8  ZF Commercial Vehicle Control Systems India Ltd.  INE342J01019   

                     industry  quantity market_value_lakhs aum_percent  
4                 Automobiles  20096960          193754.79        5.18  
5  Chemicals & Petrochemicals   1631795          116330.67        3.11  
6    Industrial Manufacturing    298145           105901.1        2.83  
7                Construction   7900000             103095